# Exploratory Data Analysis (EDA)

## Table of Contents
1. [Dataset Overview](#dataset-overview)
2. [Handling Missing Values](#handling-missing-values)
3. [Feature Distributions](#feature-distributions)
4. [Possible Biases](#possible-biases)
5. [Correlations](#correlations)

In [ ]:
# Import necessary libraries
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## Dataset Overview

This dataset contains scanning electron microscope (SEM) images of MEMS devices and electrodes. The original images were manually categorized into 8 classes: `3d_edge`, `Bond-Pad-Array`, `cantilever`, `close_up_line`, `Electrode`, `label`, `microfluidic`, and `waveguide`.

Each original image contains a gray SEM region and a white microscope information ribbon. During preprocessing, the white ribbon was cropped away so the baseline model trains only on the real SEM image region. The cropped images were resized to 224 x 224 RGB pixels for the pretrained Microsoft ResNet-50 model.

The baseline dataset used for modeling contains 3,422 prepared image samples: 2,867 training images and 555 validation images. Each sample has 150,528 pixel-level input features, calculated as 224 x 224 x 3. The target variable is the image class label.

In [ ]:
# Load the prepared dataset manifests
prepared_dir = Path(r"C:/Users/kom-e14-1/Documents/Codex/2026-06-05/i-have-sem-images-i-chategorized/outputs/prepared_sem_dataset")

train_df = pd.read_csv(prepared_dir / "train_manifest.csv")
val_df = pd.read_csv(prepared_dir / "val_manifest.csv")
df = pd.concat([train_df, val_df], ignore_index=True)

# Number of samples
num_samples = df.shape[0]

# Number of model input features: 224 x 224 x 3 RGB pixels
num_features = 224 * 224 * 3

print(f"Number of samples: {num_samples}")
print(f"Number of model input features per image: {num_features}")
print(f"Number of classes: {df['label'].nunique()}")

# Display the first few rows of the dataframe to show the structure
print("Example data:")
display(df.head())

## Handling Missing Values

The main modeling fields are `image_path`, `label`, and `label_id`. These fields should not contain missing values because every model sample needs an image and a class label. Some metadata fields may contain missing values because OCR extraction from the microscope ribbon is optional and was not used in the baseline image-only model.

In [ ]:
# Check for missing values
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# Handling missing values
# For the baseline ResNet model, no imputation is needed because only image_path and label are used.
# Metadata columns with missing OCR values are kept as missing because they are not used in the baseline model.

required_columns = ["image_path", "label", "label_id"]
print(df[required_columns].isnull().sum())

## Feature Distributions

For this image classification project, the most important distribution is the target label distribution. The original dataset was imbalanced, so minority classes were augmented in the training set. Validation data was not augmented, which gives a more realistic estimate of model performance.

In [ ]:
# Plot class distribution after preprocessing
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="label", hue="split", order=sorted(df["label"].unique()))
plt.title("Prepared Dataset Label Distribution by Split")
plt.xlabel("Class label")
plt.ylabel("Number of images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Possible Biases

The main bias risk is class imbalance. The classes `Bond-Pad-Array`, `close_up_line`, and `Electrode` have many more original samples than `label`, `microfluidic`, and `cantilever`. This can cause the model to perform better on majority classes and worse on minority classes. Augmentation was used to reduce imbalance in the training set, but augmentation cannot fully replace collecting more real images for small classes.

In [ ]:
# Check class imbalance in train and validation splits
class_balance = df.groupby(["split", "label"]).size().reset_index(name="count")
display(class_balance)

plt.figure(figsize=(10, 5))
sns.barplot(data=class_balance, x="label", y="count", hue="split")
plt.title("Class Balance After Preprocessing")
plt.xlabel("Class label")
plt.ylabel("Number of images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Correlations

The baseline model uses image pixels, so ordinary tabular correlation analysis is limited. However, the extracted microscope metadata can be explored separately. These metadata fields were saved for future work and may later be combined with image features.

In [ ]:
# Explore correlations among numeric metadata features, if available
metadata_path = prepared_dir / "metadata_features.csv"

if metadata_path.exists():
    metadata_df = pd.read_csv(metadata_path)
    numeric_metadata = metadata_df[[
        col for col in ["scale_value", "eht_kv", "wd_mm", "stage_t_deg", "stage_z_mm", "mag_value", "aperture_um"]
        if col in metadata_df.columns
    ]].apply(pd.to_numeric, errors="coerce")
    
    if numeric_metadata.dropna(how="all").shape[1] > 1:
        correlation_matrix = numeric_metadata.corr()
        plt.figure(figsize=(8, 6))
        sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
        plt.title("Correlation Between Extracted Metadata Features")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough numeric metadata values were available for correlation analysis.")
else:
    print("metadata_features.csv was not found.")